In [12]:
# =============================================================================
# Zelle 01 – Setup & Daten laden (Modelltraining Modell B)
# =============================================================================
import sys
sys.path.append('../src')

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from viz_config_v2 import apply_store44_style, save_figure, COLOR_GOLD, COLOR_BLUE, COLOR_GREEN, COLOR_TEXT_MUTED, BOXPLOT_STYLE
from preprocessing import load_dataset_b, FEATURE_SETS_B, WandstaerkeDNResidualizer, baue_preprocessing_pipeline_b, X_B_MERKMALE_ENCODED, Y_B_MERKMALE

SEED = 42
apply_store44_style()

df_b = load_dataset_b("../data/processed/model_b_preprocessed.csv")

print(f"Datensatz: {df_b.shape}")
print(f"X_B (encoded, {len(X_B_MERKMALE_ENCODED)}): {X_B_MERKMALE_ENCODED}")
print(f"Y_B ({len(Y_B_MERKMALE)}): {Y_B_MERKMALE}")
print(f"Feature-Set-Kandidaten: {list(FEATURE_SETS_B.keys())}")

Datensatz: (2000, 14)
X_B (encoded, 7): ['material_mfr', 'dn_ziel', 'wandstaerke_soll', 'dickentoleranz', 'produktionsgeschwindigkeit_soll', 'ovalitaet_anforderung', 'wandtyp_einwandig']
Y_B (6): ['schneckendrehzahl', 'massetemperatur', 'duesenspalt', 'vakuumniveau', 'innenluftdruck', 'kuehlwassertemperatur']
Feature-Set-Kandidaten: ['original', 'nur_dn', 'nur_wandstaerke', 'residualisiert']


In [13]:
# =============================================================================
# Zelle 02 – Train/Test-Split (einmalig fixiert, fuer alle Kombinationen)
# =============================================================================
# Analog Modell A: ein einziger Split, konsistent fuer alle Feature-Sets
# und Modellkombinationen verwendet. Kein Stratify noetig (keine
# Klassifikation, alle Y_B kontinuierlich).
# =============================================================================
from sklearn.model_selection import train_test_split, KFold

train_idx, test_idx = train_test_split(df_b.index, test_size=0.2, random_state=SEED)

print(f"Train: {len(train_idx)} Zeilen, Test: {len(test_idx)} Zeilen")

# --- Aeussere 5-Fold-CV, konsistent fuer alle Kombinationen ---
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
fold_splits = list(kf.split(train_idx))
fold_splits_idx = [(train_idx[tr_pos], train_idx[val_pos]) for tr_pos, val_pos in fold_splits]

print(f"Folds: {len(fold_splits_idx)}")
print(f"Beispiel Fold 0: Train={len(fold_splits_idx[0][0])}, Val={len(fold_splits_idx[0][1])}")

Train: 1600 Zeilen, Test: 400 Zeilen
Folds: 5
Beispiel Fold 0: Train=1280, Val=320


In [14]:
# =============================================================================
# Zelle 02b – Verteilungspruefung: Y_B in Train/Test/Folds vergleichbar?
# =============================================================================
# Bei kontinuierlichen Zielgroessen gibt es keine automatische Stratifizierung
# (im Gegensatz zu Klassifikation) - muss explizit geprueft werden, ob
# Zufalls-Split zu verzerrten Teilmengen fuehrt.
# =============================================================================

print("=== Train vs. Test: Mittelwert und Std je Y_B-Groesse ===")
vergleich_train_test = []
for y_col in Y_B_MERKMALE:
    train_werte = df_b.loc[train_idx, y_col]
    test_werte = df_b.loc[test_idx, y_col]
    vergleich_train_test.append({
        "y_merkmal": y_col,
        "train_mean": round(train_werte.mean(), 2), "test_mean": round(test_werte.mean(), 2),
        "train_std": round(train_werte.std(), 2), "test_std": round(test_werte.std(), 2),
        "differenz_mean_prozent": round(abs(train_werte.mean()-test_werte.mean())/train_werte.mean()*100, 2),
    })
vergleich_df_split = pd.DataFrame(vergleich_train_test)
print(vergleich_df_split.to_string(index=False))

print("\n=== Verteilung je Fold (Mittelwert), alle 5 Folds ===")
fold_vergleich = []
for fold_num, (tr_idx, val_idx) in enumerate(fold_splits_idx):
    for y_col in Y_B_MERKMALE:
        fold_vergleich.append({
            "fold": fold_num, "y_merkmal": y_col,
            "val_mean": round(df_b.loc[val_idx, y_col].mean(), 2),
            "val_std": round(df_b.loc[val_idx, y_col].std(), 2),
        })
fold_df = pd.DataFrame(fold_vergleich)
fold_pivot = fold_df.pivot(index="y_merkmal", columns="fold", values="val_mean")
print(fold_pivot.to_string())

fold_pivot.to_csv("../reports/tables/11_fold_verteilung_check_model_b.csv")
vergleich_df_split.to_csv("../reports/tables/11_train_test_verteilung_check_model_b.csv", index=False)

=== Train vs. Test: Mittelwert und Std je Y_B-Groesse ===
            y_merkmal  train_mean  test_mean  train_std  test_std  differenz_mean_prozent
    schneckendrehzahl       44.08      44.33       7.32      7.54                    0.57
      massetemperatur      204.33     204.30       5.80      5.80                    0.01
          duesenspalt        2.26       2.26       0.64      0.71                    0.19
         vakuumniveau       97.81      98.53      17.68     17.34                    0.74
       innenluftdruck       80.00      79.40      16.79     18.08                    0.75
kuehlwassertemperatur       19.05      19.11       2.89      2.99                    0.32

=== Verteilung je Fold (Mittelwert), alle 5 Folds ===
fold                        0       1       2       3       4
y_merkmal                                                    
duesenspalt              2.20    2.26    2.19    2.35    2.28
innenluftdruck          79.92   79.54   81.85   79.05   79.63
kuehlwass

In [15]:
# =============================================================================
# Zelle 03 – Modell-Registry: 8 Modelltypen fuer Multi-Output-Regression
# =============================================================================
# Konservative Standard-Hyperparameter (kein Tuning in diesem AP, analog
# Modell A AP 3.4). random_state=SEED durchgaengig.
# =============================================================================
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.multioutput import MultiOutputRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# --- Basis-Modelle (werden je nach Bedarf pur oder mit MultiOutputRegressor-Wrapper verwendet) ---
MODELLE_B = {
    "Ridge": {"modell": Ridge(random_state=SEED), "nativ_multioutput": True},
    "RandomForest": {"modell": RandomForestRegressor(n_estimators=200, max_depth=8, min_samples_leaf=3, random_state=SEED), "nativ_multioutput": True},
    "kNN": {"modell": KNeighborsRegressor(n_neighbors=5), "nativ_multioutput": True},
    "MLP": {"modell": MLPRegressor(hidden_layer_sizes=(50,), max_iter=1000, early_stopping=True, random_state=SEED), "nativ_multioutput": True},
    "SVR": {"modell": SVR(C=1.0, kernel="rbf"), "nativ_multioutput": False},
    "HistGradientBoosting": {"modell": HistGradientBoostingRegressor(max_iter=50, max_depth=3, max_leaf_nodes=15, min_samples_leaf=20, l2_regularization=1.0, random_state=SEED), "nativ_multioutput": False},
    "XGBoost": {"modell": XGBRegressor(n_estimators=50, max_depth=3, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8, reg_lambda=2.0, random_state=SEED), "nativ_multioutput": False},
    "LightGBM": {"modell": LGBMRegressor(n_estimators=50, num_leaves=7, max_depth=3, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8, min_child_samples=30, random_state=SEED, verbose=-1), "nativ_multioutput": False},
}

print("Modell-Registry (8 Typen):")
for name, konfig in MODELLE_B.items():
    print(f"  {name:22s} nativ_multioutput={konfig['nativ_multioutput']}")

Modell-Registry (8 Typen):
  Ridge                  nativ_multioutput=True
  RandomForest           nativ_multioutput=True
  kNN                    nativ_multioutput=True
  MLP                    nativ_multioutput=True
  SVR                    nativ_multioutput=False
  HistGradientBoosting   nativ_multioutput=False
  XGBoost                nativ_multioutput=False
  LightGBM               nativ_multioutput=False


In [16]:
# =============================================================================
# Zelle 04 – Metrik-Funktion: vollstaendiges Set (bestaetigt durch Nutzer)
# =============================================================================
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, median_absolute_error

def berechne_metriken_b(y_true, y_pred, y_true_std):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    medae = median_absolute_error(y_true, y_pred)
    rmse_normiert = rmse / y_true_std if y_true_std > 0 else np.nan
    mae_normiert = mae / y_true_std if y_true_std > 0 else np.nan
    return {
        "mae": mae, "rmse": rmse, "r2": r2, "median_ae": medae,
        "rmse_normiert": rmse_normiert, "mae_normiert": mae_normiert,
    }

print("Metrik-Funktion definiert: mae, rmse, r2, median_ae, rmse_normiert, mae_normiert")
print("Train-CV-Gap, Fold-Std, Fit-Zeit, Predict-Zeit: in Ablationsschleife (Zelle 05)")
print("Prediction Interval Coverage: separater Zusatzschritt nur fuer RandomForest/Ridge/kNN (spaeter)")

Metrik-Funktion definiert: mae, rmse, r2, median_ae, rmse_normiert, mae_normiert
Train-CV-Gap, Fold-Std, Fit-Zeit, Predict-Zeit: in Ablationsschleife (Zelle 05)
Prediction Interval Coverage: separater Zusatzschritt nur fuer RandomForest/Ridge/kNN (spaeter)


In [19]:
# =============================================================================
# Zelle 05 – Ablationsschleife: 4 Feature-Sets x 8 Modelle x Struktur-Varianten
# =============================================================================
# Try/Except + Zwischenspeicherung je Kombination (bewaehrtes Robustheits-
# muster aus Modell A). Fuer nativ-multioutput-faehige Modelle werden BEIDE
# Strukturvarianten (Multi-Output vs. 6 Einzelmodelle) getestet.
# =============================================================================
from sklearn.base import clone
import warnings
warnings.filterwarnings("ignore")

OUTPUT_CSV_B = "../reports/tables/11_ablation_results_model_b.csv"

y_std_je_merkmal = {y: df_b.loc[train_idx, y].std() for y in Y_B_MERKMALE}

kombinationen = []
for modell_name, konfig in MODELLE_B.items():
    for struktur in (["multioutput", "einzeln"] if konfig["nativ_multioutput"] else ["einzeln"]):
        for fs_name in FEATURE_SETS_B:
            kombinationen.append((modell_name, struktur, fs_name))

ergebnisse_b = []
start_gesamt = time.time()

for kombi_idx, (modell_name, struktur, fs_name) in enumerate(kombinationen):
    eintrag = {"modell": modell_name, "struktur": struktur, "feature_set": fs_name, "status": "ok"}
    fold_metriken_je_y = {y: [] for y in Y_B_MERKMALE}
    fold_metriken_train_je_y = {y: [] for y in Y_B_MERKMALE}
    fold_fit_zeiten, fold_predict_zeiten = [], []

    try:
        for tr_idx, val_idx in fold_splits_idx:
            prep = baue_preprocessing_pipeline_b(fs_name)
            X_tr = prep.fit_transform(df_b.loc[tr_idx])
            X_val = prep.transform(df_b.loc[val_idx])
            y_tr = df_b.loc[tr_idx, Y_B_MERKMALE]
            y_val = df_b.loc[val_idx, Y_B_MERKMALE]

            t0 = time.time()
            if struktur == "multioutput":
                modell = clone(MODELLE_B[modell_name]["modell"])
                modell.fit(X_tr, y_tr)
                fit_zeit = time.time() - t0
                t1 = time.time()
                y_pred_val = pd.DataFrame(modell.predict(X_val), columns=Y_B_MERKMALE, index=val_idx)
                y_pred_train = pd.DataFrame(modell.predict(X_tr), columns=Y_B_MERKMALE, index=tr_idx)
                predict_zeit = time.time() - t1
            else:
                y_pred_val = pd.DataFrame(index=val_idx, columns=Y_B_MERKMALE, dtype=float)
                y_pred_train = pd.DataFrame(index=tr_idx, columns=Y_B_MERKMALE, dtype=float)
                predict_zeit_summe = 0
                for y_col in Y_B_MERKMALE:
                    modell_einzeln = clone(MODELLE_B[modell_name]["modell"])
                    modell_einzeln.fit(X_tr, y_tr[y_col])
                    t1 = time.time()
                    y_pred_val[y_col] = modell_einzeln.predict(X_val)
                    y_pred_train[y_col] = modell_einzeln.predict(X_tr)
                    predict_zeit_summe += time.time() - t1
                fit_zeit = time.time() - t0 - predict_zeit_summe
                predict_zeit = predict_zeit_summe

            fold_fit_zeiten.append(fit_zeit)
            fold_predict_zeiten.append(predict_zeit)

            for y_col in Y_B_MERKMALE:
                m_val = berechne_metriken_b(y_val[y_col], y_pred_val[y_col], y_std_je_merkmal[y_col])
                m_train = berechne_metriken_b(y_tr[y_col], y_pred_train[y_col], y_std_je_merkmal[y_col])
                fold_metriken_je_y[y_col].append(m_val)
                fold_metriken_train_je_y[y_col].append(m_train)

        for y_col in Y_B_MERKMALE:
            for metrik_key in ["mae", "rmse", "r2", "median_ae", "rmse_normiert", "mae_normiert"]:
                werte_val = [fm[metrik_key] for fm in fold_metriken_je_y[y_col]]
                werte_train = [fm[metrik_key] for fm in fold_metriken_train_je_y[y_col]]
                eintrag[f"{y_col}_{metrik_key}_mean"] = np.nanmean(werte_val)
                eintrag[f"{y_col}_{metrik_key}_std"] = np.nanstd(werte_val)
                eintrag[f"{y_col}_{metrik_key}_gap"] = np.nanmean(werte_train) - np.nanmean(werte_val)

        eintrag["fit_zeit_sekunden_mean"] = np.mean(fold_fit_zeiten)
        eintrag["predict_zeit_sekunden_mean"] = np.mean(fold_predict_zeiten)
        eintrag["r2_durchschnitt_alle_y"] = np.mean([eintrag[f"{y}_r2_mean"] for y in Y_B_MERKMALE])

    except Exception as e:
        eintrag["status"] = "fehlgeschlagen"
        eintrag["fehler"] = f"{type(e).__name__}: {str(e)[:300]}"

    ergebnisse_b.append(eintrag)
    pd.DataFrame(ergebnisse_b).to_csv(OUTPUT_CSV_B, index=False)
    print(f"[{kombi_idx+1}/{len(kombinationen)}] {modell_name:22s} {struktur:11s} {fs_name:16s} -> {eintrag['status']}")

print(f"\nGesamtzeit: {time.time()-start_gesamt:.1f}s")
ergebnisse_b_df = pd.DataFrame(ergebnisse_b)
print(f"Erfolgreich: {(ergebnisse_b_df['status']=='ok').sum()} / {len(ergebnisse_b_df)}")

[1/48] Ridge                  multioutput original         -> ok
[2/48] Ridge                  multioutput nur_dn           -> ok
[3/48] Ridge                  multioutput nur_wandstaerke  -> ok
[4/48] Ridge                  multioutput residualisiert   -> ok
[5/48] Ridge                  einzeln     original         -> ok
[6/48] Ridge                  einzeln     nur_dn           -> ok
[7/48] Ridge                  einzeln     nur_wandstaerke  -> ok
[8/48] Ridge                  einzeln     residualisiert   -> ok
[9/48] RandomForest           multioutput original         -> ok
[10/48] RandomForest           multioutput nur_dn           -> ok
[11/48] RandomForest           multioutput nur_wandstaerke  -> ok
[12/48] RandomForest           multioutput residualisiert   -> ok
[13/48] RandomForest           einzeln     original         -> ok
[14/48] RandomForest           einzeln     nur_dn           -> ok
[15/48] RandomForest           einzeln     nur_wandstaerke  -> ok
[16/48] RandomFores